In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('/projects/activities/kappsen-tmc/USERS/domans/differential-annotator-dev/DIANNE')
import dianne
import viewer
dianne.setNotebookWidth(100)

In [ ]:
dataPath = '/projects/activities/kappsen-tmc/USERS/domans/wsi-viewer/data/'

# Download the demo data from Zenodo and unzip it to the data folder
dianne.downloadZIPFromZenodo(dataPath, 'https://zenodo.org/records/19225051/files/', 'RMS2194.oid0.zip')

# Load the data and prepare the patches for annotation and training the classifier
samples  = ['RMS2194.oid0']
classifierPaths = 'classifiers/'
dianne.setupClassifierPaths(classifierPaths)
fname = 'features/false-1-ctranspath_features.tsv.gz'

ts, mpp = 112., 0.25 # center-to-center tile distance in um and pixel size
patch_size = 8 # number of tiles, in each dimension, to include in a patch (e.g. 8 means 8x8=64 tiles per patch)
ads, imgs, patchCoordinates, patchesCDFs, qs, ts, mpp, L, N = dianne.loadDataAndPreparePatches(samples, dataPath, fname, L=None, ts=ts, mpp=mpp, N=patch_size)

In [ ]:
clicks, strokes, stop = viewer.create_viewer(f"{dataPath}/{samples[0]}/image.ome.tiff", height="700px")

In [ ]:
# Train the classifier on the patches and the corresponding features
body_overlap = 0.25 # Fraction overlap between the drawn annotations and the tiles
clf, patchesCDFsMod, annotationsMod = dianne.getClassifierForFromStrokes(strokes, patchCoordinates, 224, body_overlap, patch_size,
                                                                        ads, samples, qs, augFunc=dianne.PCMA, alpha=0.8, seed=0)
# Run inference and visualize the results
if not clf is None:
    multiplier = 4 # Interpolation parameter for the probability heatmap
    x, y, p = dianne.inferProb(ads[samples[0]], clf, qs, tsize=ts/mpp, R=2, erode=False)
    xi, yi, pi = dianne.interpolatePoints(x, y, p, multiplier=multiplier)
    dianne.showProbImg(xi, yi, pi, f=2, figsize=(3, 3), ts=ts, mpp=mpp, title=samples[0], invert=False)
    viewer.set_overlay_points(xi, yi, pi, delta=448 / multiplier, alpha=0.5, color_low='#FFA500', color_high='#0000FF')

In [ ]:
# Create probability masks, extract contours and visualize them on the H&E image
downsampled_map, fshape = dianne.makeProbMask(ads[samples[0]], imgs[samples[0]], x, y, p, ts=ts, mpp=mpp, downfactor=16, verbose=True)
geojson = dianne.extractContoursForQuPath(downsampled_map, fshape, cutoff=0.8, min_area=10**6, downfactor=16, sigma=100)
dianne.viewContoursOnImage(imgs[samples[0]], geojson, fshape, level=2, figsize=(12, 12), linewidth=1)